# 06 Test Set Evaluation

## Purpose

This notebook runs held-out test evaluation for one completed training run.

## Inputs

- `best_model/` from one experiment folder in `MyDrive/ProjectRoot2/checkpoints/`
- `test_dataset.pt` from the matching tokenizer setting in `MyDrive/ProjectRoot2/tokenized_datasets/`
- `MyDrive/ProjectRoot2/registry/run_index.csv`

## Outputs

- summary test metrics
- per-class metrics
- ROC plot and ROC summary table
- precision-recall plot and PR summary table
- qualitative probe results
- run index updates

## Notes to myself

I'm using the test split here because this is where I want the real comparison between tokenizers and model runs to happen. The validation split is only for training diagnostics.


## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- test outputs stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

# Mount Drive so this runtime can read saved datasets and write test outputs.
drive.mount('/content/drive')

# Define the private GitHub repo used for the rebuilt workflow.
GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta-sandbox2'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone the repo into the runtime if needed. Otherwise update the existing clone.
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

# Add the repo to sys.path so imports from src/ work in Colab.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


In [ ]:
# ==============================================================================
# 1. TEST EVALUATION CONFIG
# ==============================================================================
TOKENIZER_FAMILY = 'hybrid_char_bpe'
SETTING_LABEL = 'v70_m2'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv70_m2'

MLM_PROBABILITY = 0.15
MASK_SEED = 42
BATCH_SIZE = 16


In [ ]:
# ==============================================================================
# 2. IMPORTS
# ==============================================================================
import json
import os
import sys

import pandas as pd
import torch
from IPython.display import display

sys.path.append(REPO_DIR)

from src.run_index import upsert_run_record
from src.test_evaluation import (
    build_masked_test_dataset,
    build_masking_summary,
    build_test_summary_row,
    compute_per_class_metrics,
    compute_summary_classification_metrics,
    compute_topk_metrics,
    get_core_probe_cases,
    get_tokenizer_specific_roc_pr_tokens,
    load_test_artifacts,
    plot_pr_for_selected_classes,
    plot_roc_for_selected_classes,
    run_mlm_test_predictions,
    run_structured_qualitative_probe,
)


In [ ]:
# ==============================================================================
# 3. PATHS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'
RUN_INDEX_PATH = os.path.join(PROJECT_ROOT, 'registry', 'run_index.csv')

BEST_MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    'checkpoints',
    TOKENIZER_FAMILY,
    EXPERIMENT_NAME,
    'best_model',
)

TEST_DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    'tokenized_datasets',
    TOKENIZER_FAMILY,
    SETTING_LABEL,
    'test_dataset.pt',
)

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    'results',
    'test_evaluation',
    TOKENIZER_FAMILY,
    EXPERIMENT_NAME,
)
os.makedirs(RESULTS_DIR, exist_ok=True)

TEST_SUMMARY_JSON_PATH = os.path.join(RESULTS_DIR, 'test_summary.json')
TEST_SUMMARY_ROW_PATH = os.path.join(RESULTS_DIR, 'test_summary_row.csv')
MASKING_SUMMARY_PATH = os.path.join(RESULTS_DIR, 'masking_summary.csv')
PER_CLASS_METRICS_PATH = os.path.join(RESULTS_DIR, 'per_class_metrics.csv')
ROC_PLOT_PATH = os.path.join(RESULTS_DIR, 'roc_curves.png')
ROC_SUMMARY_PATH = os.path.join(RESULTS_DIR, 'roc_auc_summary.csv')
PR_PLOT_PATH = os.path.join(RESULTS_DIR, 'pr_curves.png')
PR_SUMMARY_PATH = os.path.join(RESULTS_DIR, 'pr_auc_summary.csv')
QUALITATIVE_PROBE_PATH = os.path.join(RESULTS_DIR, 'qualitative_probe_results.csv')

print(f'Best model dir: {BEST_MODEL_DIR}')
print(f'Test dataset path: {TEST_DATASET_PATH}')
print(f'Results dir: {RESULTS_DIR}')


In [ ]:
# ==============================================================================
# 4. CHECK INPUT ARTIFACTS
# ==============================================================================
required_paths = [BEST_MODEL_DIR, TEST_DATASET_PATH]
for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing required path: {path}')

print('Required artifacts found.')


In [ ]:
# ==============================================================================
# 5. LOAD MODEL, TOKENIZER, AND TEST DATA
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tokenizer, model, test_dataset = load_test_artifacts(
    model_dir=BEST_MODEL_DIR,
    test_dataset_path=TEST_DATASET_PATH,
    device=device,
)

print(f'Loaded tokenizer from: {BEST_MODEL_DIR}')
print(f'Loaded test dataset with {len(test_dataset)} sequences')


In [ ]:
# ==============================================================================
# 6. BUILD DETERMINISTIC MASKED TEST SET
# ==============================================================================
masked_dataset_dict = build_masked_test_dataset(
    test_dataset=test_dataset,
    tokenizer=tokenizer,
    mlm_probability=MLM_PROBABILITY,
    seed=MASK_SEED,
)

masking_summary = build_masking_summary(
    masked_dataset_dict=masked_dataset_dict,
    mlm_probability=MLM_PROBABILITY,
    mask_seed=MASK_SEED,
)

display(masking_summary)
masking_summary.to_csv(MASKING_SUMMARY_PATH, index=False)

print(f'Masking summary saved to: {MASKING_SUMMARY_PATH}')


In [ ]:
# ==============================================================================
# 7. RUN TEST PREDICTIONS
# ==============================================================================
y_true, y_pred, y_probs, sequence_top1_flags, sequence_top3_flags = run_mlm_test_predictions(
    model=model,
    masked_dataset_dict=masked_dataset_dict,
    batch_size=BATCH_SIZE,
    device=device,
)

prediction_summary = pd.DataFrame(
    {
        'metric': [
            'masked_token_predictions',
            'evaluated_sequences',
        ],
        'value': [
            len(y_true),
            len(sequence_top1_flags),
        ],
    }
)

display(prediction_summary)


In [ ]:
# ==============================================================================
# 8. COMPUTE SUMMARY METRICS
# ==============================================================================
topk_metrics = compute_topk_metrics(
    y_true=y_true,
    y_probs=y_probs,
    y_pred=y_pred,
    sequence_top1_flags=sequence_top1_flags,
    sequence_top3_flags=sequence_top3_flags,
)

classification_metrics = compute_summary_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
)

all_metrics = {}
all_metrics.update(topk_metrics)
all_metrics.update(classification_metrics)

summary_df = pd.DataFrame(
    {
        'metric': list(all_metrics.keys()),
        'value': list(all_metrics.values()),
    }
)

display(summary_df)

with open(TEST_SUMMARY_JSON_PATH, 'w', encoding='utf-8') as file:
    json.dump(all_metrics, file, indent=2)

test_summary_row = build_test_summary_row(
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    experiment_name=EXPERIMENT_NAME,
    metrics_dict=all_metrics,
)
test_summary_row.to_csv(TEST_SUMMARY_ROW_PATH, index=False)

print(f'Test summary JSON saved to: {TEST_SUMMARY_JSON_PATH}')
print(f'Test summary row saved to: {TEST_SUMMARY_ROW_PATH}')


In [ ]:
# ==============================================================================
# 9. PER-CLASS METRICS
# ==============================================================================
per_class_metrics = compute_per_class_metrics(
    y_true=y_true,
    y_pred=y_pred,
    tokenizer=tokenizer,
)

display(per_class_metrics.head(20))
per_class_metrics.to_csv(PER_CLASS_METRICS_PATH, index=False)

print(f'Per-class metrics saved to: {PER_CLASS_METRICS_PATH}')


In [ ]:
# ==============================================================================
# 10. ROC CURVES FOR TOKENIZER-SPECIFIC TOKENS
# ==============================================================================
roc_pr_tokens = get_tokenizer_specific_roc_pr_tokens()
selected_tokens = roc_pr_tokens[TOKENIZER_FAMILY]

print('Selected ROC/PR tokens:')
print(selected_tokens)

roc_summary = plot_roc_for_selected_classes(
    y_true=y_true,
    y_probs=y_probs,
    tokenizer=tokenizer,
    selected_tokens=selected_tokens,
    save_path=ROC_PLOT_PATH,
)

display(roc_summary)
roc_summary.to_csv(ROC_SUMMARY_PATH, index=False)

print(f'ROC plot saved to: {ROC_PLOT_PATH}')
print(f'ROC summary saved to: {ROC_SUMMARY_PATH}')


In [ ]:
# ==============================================================================
# 11. PRECISION-RECALL CURVES FOR TOKENIZER-SPECIFIC TOKENS
# ==============================================================================
pr_summary = plot_pr_for_selected_classes(
    y_true=y_true,
    y_probs=y_probs,
    tokenizer=tokenizer,
    selected_tokens=selected_tokens,
    save_path=PR_PLOT_PATH,
)

display(pr_summary)
pr_summary.to_csv(PR_SUMMARY_PATH, index=False)

print(f'PR plot saved to: {PR_PLOT_PATH}')
print(f'PR summary saved to: {PR_SUMMARY_PATH}')


In [ ]:
# ==============================================================================
# 12. QUALITATIVE PROBES
# ==============================================================================
probe_cases = get_core_probe_cases()

qualitative_probe_results = run_structured_qualitative_probe(
    model=model,
    tokenizer=tokenizer,
    tokenizer_family=TOKENIZER_FAMILY,
    probe_cases=probe_cases,
    device=device,
)

display(qualitative_probe_results)
qualitative_probe_results.to_csv(QUALITATIVE_PROBE_PATH, index=False)

print(f'Qualitative probe results saved to: {QUALITATIVE_PROBE_PATH}')


In [ ]:
# ==============================================================================
# 13. UPDATE RUN INDEX
# ==============================================================================
upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'results_dir': RESULTS_DIR,
        'test_metrics_path': TEST_SUMMARY_JSON_PATH,
        'qualitative_probe_path': QUALITATIVE_PROBE_PATH,
        'notebook_used': 'notebooks/06_test_set_evaluation2.ipynb',
        'run_status': 'tested',
        'notes': '',
    },
)

print(f'Run index updated: {RUN_INDEX_PATH}')


## GitHub sync note

Same idea as the earlier notebooks. The test outputs stay in Drive. The notebook itself stays versioned in GitHub.


In [ ]:
# ==============================================================================
# 14. SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks', '06_test_set_evaluation2.ipynb')
DRIVE_NOTEBOOK_PATH = '/content/drive/MyDrive/Colab Notebooks/06_test_set_evaluation2.ipynb'

if os.path.exists(DRIVE_NOTEBOOK_PATH):
    !cp "{DRIVE_NOTEBOOK_PATH}" "{REPO_NOTEBOOK_PATH}"

    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/06_test_set_evaluation2.ipynb src/test_evaluation.py
    !git commit -m "Update 06_test_set_evaluation2" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
else:
    print(f'Notebook file not found at: {DRIVE_NOTEBOOK_PATH}')
    print('Save the notebook in Colab, then run this cell again.')
